<a href="https://colab.research.google.com/github/prometricas/William_Rondon/blob/main/PSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculos inferenciales PSP – sección 4.4.2

Notebook para obtener las tablas estadísticas del análisis inferencial en el aprendizaje del Sistema Periódico. No genera gráficos; solo produce tablas para normalidad, comparación entre grupos, cambios intragrupo y contraste de cambios.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display

# Yo cargo el archivo validado; si no está en Colab, yo pido subirlo.
archivo = "data_Quimica_validado.xlsx"
try:
    psp = pd.read_excel(archivo, sheet_name="PSP")
except FileNotFoundError:
    from google.colab import files
    subidos = files.upload()
    archivo = list(subidos.keys())[0]
    psp = pd.read_excel(archivo, sheet_name="PSP")

# Yo verifico la cantidad de casos por grupo y momento.
print(pd.crosstab(psp["grupo"], psp["momento"]))
psp.head()

Saving data_Quimica_validado.xlsx to data_Quimica_validado.xlsx
momento       Postest  Pretest
grupo                         
Control            23       23
Experimental       27       27


,momento,grupo,curso,id_estudiante,nombre_estudiante,sexo,edad,n_items_validos,psp_total,psp_proporcion_maxima,...,i13_puntaje,i13_patron,i14_n1,i14_n2,i14_puntaje,i14_patron,i15_n1,i15_n2,i15_puntaje,i15_patron
0,Pretest,Control,1004,B_Est 1,Hanna Guarumo,F,16,15,8,0.266667,...,1,Razonamiento Emergente,A,2,0,Incorrecto,A,2,2,Correcto
1,Postest,Control,1004,B_Est 1,Hanna Guarumo,F,16,15,14,0.466667,...,0,Incorrecto,B,3,2,Correcto,A,2,2,Correcto
2,Pretest,Control,1004,B_Est 2,Ana Mileth Collo Valencia,F,16,15,9,0.300000,...,0,Incorrecto,A,1,0,Incorrecto,A,1,1,Falso +
3,Postest,Control,1004,B_Est 2,Ana Mileth Collo Valencia,F,16,15,11,0.366667,...,1,Falso +,B,3,2,Correcto,A,2,2,Correcto
4,Pretest,Control,1004,B_Est 3,Pineda Caro Liz,F,15,15,25,0.833333,...,2,Correcto,B,3,2,Correcto,B,1,0,Incorrecto


In [2]:
# Yo defino únicamente las variables necesarias para la sección 4.4.2.
variables = {
    "Concepto de elemento químico": "d1_concepto_elemento_quimico_media",
    "Criterios de ordenamiento del sistema periódico": "d2_criterios_ordenamiento_sp_media",
    "Periodicidad química": "d3_periodicidad_quimica_media",
    "Interpretación del sistema periódico en función de la estructura atómica": "d4_estructura_atomica_sp_media",
    "Naturaleza de la Ciencia en relación con el sistema periódico": "d5_ndc_sistema_periodico_media",
    "Puntaje total de la Prueba de Aprendizaje del Sistema Periódico": "psp_total"
}

grupos = ["Control", "Experimental"]
momentos = ["Pretest", "Postest"]

# Yo dejo funciones breves para redondear, calcular rangos y obtener tamaños del efecto.
def r3(x):
    return np.nan if pd.isna(x) else round(float(x), 3)

def p3(p):
    if pd.isna(p):
        return ""
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def rangos_medios(x, y):
    valores = np.r_[x, y]
    rangos = stats.rankdata(valores, method="average")
    return rangos[:len(x)].mean(), rangos[len(x):].mean()

def mann_whitney(x, y):
    x = np.asarray(x)
    y = np.asarray(y)
    n1, n2 = len(x), len(y)
    resultado = stats.mannwhitneyu(x, y, alternative="two-sided", method="asymptotic", use_continuity=True)
    u_x = resultado.statistic
    u_y = n1 * n2 - u_x
    u = min(u_x, u_y)
    rango_x, rango_y = rangos_medios(x, y)
    z_abs = 0 if resultado.pvalue >= 1 else abs(stats.norm.ppf(resultado.pvalue / 2))
    z = np.sign(rango_x - rango_y) * z_abs
    r = z_abs / np.sqrt(n1 + n2)
    return u, z, resultado.pvalue, r, rango_x, rango_y

def wilcoxon_pareado(pre, post):
    pre = np.asarray(pre)
    post = np.asarray(post)
    diferencia = post - pre
    if np.all(diferencia == 0):
        return 0, 0, 1, 0
    resultado = stats.wilcoxon(post, pre, zero_method="wilcox", correction=True, alternative="two-sided", method="approx")
    z_abs = 0 if resultado.pvalue >= 1 else abs(stats.norm.ppf(resultado.pvalue / 2))
    z = np.sign(np.mean(diferencia)) * z_abs
    r = z_abs / np.sqrt(len(pre))
    return resultado.statistic, z, resultado.pvalue, r

In [3]:
def normalidad(momento):
    filas = []
    datos = psp[psp["momento"] == momento]
    for nombre, col in variables.items():
        fila = {"Variable": nombre}
        for grupo in grupos:
            x = datos.loc[datos["grupo"] == grupo, col].dropna()
            w, p = stats.shapiro(x)
            fila[f"W {grupo.lower()}"] = r3(w)
            fila[f"p {grupo.lower()}"] = p3(p)
        filas.append(fila)
    return pd.DataFrame(filas)

def comparacion_intergrupal(momento):
    filas = []
    datos = psp[psp["momento"] == momento]
    for nombre, col in variables.items():
        control = datos.loc[datos["grupo"] == "Control", col].dropna().to_numpy()
        experimental = datos.loc[datos["grupo"] == "Experimental", col].dropna().to_numpy()
        u, z, p, r, rango_control, rango_experimental = mann_whitney(control, experimental)
        filas.append({
            "Variable": nombre,
            "Rango medio control": r3(rango_control),
            "Rango medio experimental": r3(rango_experimental),
            "U de Mann-Whitney": r3(u),
            "Z": r3(z),
            "p": p3(p),
            "r": r3(r)
        })
    return pd.DataFrame(filas)

def normalidad_diferencias():
    filas = []
    for grupo in grupos:
        datos = psp[psp["grupo"] == grupo]
        for nombre, col in variables.items():
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pretest"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "postest"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            diferencia = pares["postest"] - pares["pretest"]
            w, p = stats.shapiro(diferencia)
            filas.append({"Grupo": grupo, "Variable": nombre, "W": r3(w), "p": p3(p)})
    return pd.DataFrame(filas)

def cambios_intragrupo():
    filas = []
    for grupo in grupos:
        datos = psp[psp["grupo"] == grupo]
        for nombre, col in variables.items():
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pretest"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "postest"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            w, z, p, r = wilcoxon_pareado(pares["pretest"], pares["postest"])
            filas.append({
                "Grupo": grupo,
                "Variable": nombre,
                "Media pretest": r3(pares["pretest"].mean()),
                "Media postest": r3(pares["postest"].mean()),
                "Diferencia media (postest - pretest)": r3((pares["postest"] - pares["pretest"]).mean()),
                "Estadístico de Wilcoxon": r3(w),
                "Z": r3(z),
                "p": p3(p),
                "r": r3(r)
            })
    return pd.DataFrame(filas)

def contraste_cambios():
    filas = []
    for nombre, col in variables.items():
        cambios = {}
        for grupo in grupos:
            datos = psp[psp["grupo"] == grupo]
            pre = datos.loc[datos["momento"] == "Pretest", ["id_estudiante", col]].rename(columns={col: "pretest"})
            post = datos.loc[datos["momento"] == "Postest", ["id_estudiante", col]].rename(columns={col: "postest"})
            pares = pre.merge(post, on="id_estudiante").dropna()
            cambios[grupo] = (pares["postest"] - pares["pretest"]).to_numpy()
        u, z, p, r, _, _ = mann_whitney(cambios["Experimental"], cambios["Control"])
        filas.append({
            "Variable": nombre,
            "Cambio medio experimental": r3(np.mean(cambios["Experimental"])),
            "Cambio medio control": r3(np.mean(cambios["Control"])),
            "Diferencia de cambios (experimental - control)": r3(np.mean(cambios["Experimental"]) - np.mean(cambios["Control"])),
            "U de Mann-Whitney": r3(u),
            "Z": r3(z),
            "p": p3(p),
            "r": r3(r)
        })
    return pd.DataFrame(filas)

In [4]:
# Yo genero las tablas principales de normalidad e inferencia intergrupal.
tabla_21_normalidad_pretest = normalidad("Pretest")
tabla_22_intergrupal_pretest = comparacion_intergrupal("Pretest")
tabla_23_normalidad_postest = normalidad("Postest")
tabla_24_intergrupal_postest = comparacion_intergrupal("Postest")

display(tabla_21_normalidad_pretest)
display(tabla_22_intergrupal_pretest)
display(tabla_23_normalidad_postest)
display(tabla_24_intergrupal_postest)

,Variable,W control,p control,W experimental,p experimental
0,Concepto de elemento químico,0.893,0.018,0.936,0.097
1,Criterios de ordenamiento del sistema periódico,0.940,0.179,0.865,0.002
2,Periodicidad química,0.845,0.002,0.840,< 0.001
3,Interpretación del sistema periódico en funció...,0.931,0.117,0.884,0.006
4,Naturaleza de la Ciencia en relación con el si...,0.890,0.016,0.918,0.036
5,Puntaje total de la Prueba de Aprendizaje del ...,0.954,0.350,0.957,0.307


,Variable,Rango medio control,Rango medio experimental,U de Mann-Whitney,Z,p,r
0,Concepto de elemento químico,29.043,22.481,229.0,1.619,0.105,0.229
1,Criterios de ordenamiento del sistema periódico,29.500,22.093,218.5,1.837,0.066,0.260
2,Periodicidad química,27.826,23.519,257.0,1.078,0.281,0.152
3,Interpretación del sistema periódico en funció...,26.826,24.370,280.0,0.608,0.543,0.086
4,Naturaleza de la Ciencia en relación con el si...,26.652,24.519,284.0,0.518,0.604,0.073
5,Puntaje total de la Prueba de Aprendizaje del ...,29.543,22.056,217.5,1.810,0.070,0.256


,Variable,W control,p control,W experimental,p experimental
0,Concepto de elemento químico,0.897,0.022,0.922,0.044
1,Criterios de ordenamiento del sistema periódico,0.855,0.003,0.916,0.031
2,Periodicidad química,0.924,0.080,0.924,0.049
3,Interpretación del sistema periódico en funció...,0.892,0.018,0.938,0.106
4,Naturaleza de la Ciencia en relación con el si...,0.897,0.022,0.877,0.004
5,Puntaje total de la Prueba de Aprendizaje del ...,0.901,0.027,0.949,0.197


,Variable,Rango medio control,Rango medio experimental,U de Mann-Whitney,Z,p,r
0,Concepto de elemento químico,23.957,26.815,275.0,-0.697,0.486,0.099
1,Criterios de ordenamiento del sistema periódico,23.957,26.815,275.0,-0.693,0.488,0.098
2,Periodicidad química,23.848,26.907,272.5,-0.743,0.458,0.105
3,Interpretación del sistema periódico en funció...,24.739,26.148,293.0,-0.338,0.735,0.048
4,Naturaleza de la Ciencia en relación con el si...,25.717,25.315,305.5,0.091,0.928,0.013
5,Puntaje total de la Prueba de Aprendizaje del ...,23.783,26.963,271.0,-0.762,0.446,0.108


In [5]:
# Yo genero las tablas para los cambios pretest-postest y el contraste de cambios entre grupos.
tabla_25_normalidad_diferencias = normalidad_diferencias()
tabla_26_cambios_intragrupo = cambios_intragrupo()
tabla_27_contraste_cambios = contraste_cambios()

display(tabla_25_normalidad_diferencias)
display(tabla_26_cambios_intragrupo)
display(tabla_27_contraste_cambios)

,Grupo,Variable,W,p
0,Control,Concepto de elemento químico,0.923,0.078
1,Control,Criterios de ordenamiento del sistema periódico,0.906,0.033
2,Control,Periodicidad química,0.943,0.213
3,Control,Interpretación del sistema periódico en funció...,0.954,0.360
4,Control,Naturaleza de la Ciencia en relación con el si...,0.949,0.282
5,Control,Puntaje total de la Prueba de Aprendizaje del ...,0.865,0.005
6,Experimental,Concepto de elemento químico,0.890,0.008
7,Experimental,Criterios de ordenamiento del sistema periódico,0.961,0.381
8,Experimental,Periodicidad química,0.956,0.298
9,Experimental,Interpretación del sistema periódico en funció...,0.925,0.052


,Grupo,Variable,Media pretest,Media postest,Diferencia media (postest - pretest),Estadístico de Wilcoxon,Z,p,r
0,Control,Concepto de elemento químico,1.174,1.261,0.087,94.0,0.731,0.465,0.152
1,Control,Criterios de ordenamiento del sistema periódico,0.783,1.174,0.391,55.0,2.088,0.037,0.435
2,Control,Periodicidad química,0.522,0.739,0.217,80.5,1.202,0.229,0.251
3,Control,Interpretación del sistema periódico en funció...,0.638,1.000,0.362,24.0,2.662,0.008,0.555
4,Control,Naturaleza de la Ciencia en relación con el si...,1.072,1.464,0.391,24.5,2.995,0.003,0.624
5,Control,Puntaje total de la Prueba de Aprendizaje del ...,12.565,16.913,4.348,51.5,2.423,0.015,0.505
6,Experimental,Concepto de elemento químico,0.926,1.407,0.481,21.5,3.666,< 0.001,0.705
7,Experimental,Criterios de ordenamiento del sistema periódico,0.556,1.309,0.753,6.5,4.285,< 0.001,0.825
8,Experimental,Periodicidad química,0.309,0.840,0.531,17.5,3.535,< 0.001,0.680
9,Experimental,Interpretación del sistema periódico en funció...,0.568,1.025,0.457,16.5,3.808,< 0.001,0.733


,Variable,Cambio medio experimental,Cambio medio control,Diferencia de cambios (experimental - control),U de Mann-Whitney,Z,p,r
0,Concepto de elemento químico,0.481,0.087,0.395,224.0,1.680,0.093,0.238
1,Criterios de ordenamiento del sistema periódico,0.753,0.391,0.362,238.5,1.395,0.163,0.197
2,Periodicidad química,0.531,0.217,0.313,245.0,1.272,0.203,0.180
3,Interpretación del sistema periódico en funció...,0.457,0.362,0.094,280.5,0.577,0.564,0.082
4,Naturaleza de la Ciencia en relación con el si...,0.506,0.391,0.115,277.0,0.646,0.519,0.091
5,Puntaje total de la Prueba de Aprendizaje del ...,8.185,4.348,3.837,218.0,1.797,0.072,0.254


In [6]:
# Yo exporto todas las tablas a Excel para copiarlas en Word.
salida = "tablas_PSP_4_4_2.xlsx"
with pd.ExcelWriter(salida) as writer:
    tabla_21_normalidad_pretest.to_excel(writer, sheet_name="T21_normalidad_pre", index=False)
    tabla_22_intergrupal_pretest.to_excel(writer, sheet_name="T22_inter_pre", index=False)
    tabla_23_normalidad_postest.to_excel(writer, sheet_name="T23_normalidad_post", index=False)
    tabla_24_intergrupal_postest.to_excel(writer, sheet_name="T24_inter_post", index=False)
    tabla_25_normalidad_diferencias.to_excel(writer, sheet_name="T25_normalidad_dif", index=False)
    tabla_26_cambios_intragrupo.to_excel(writer, sheet_name="T26_cambios_intra", index=False)
    tabla_27_contraste_cambios.to_excel(writer, sheet_name="T27_contraste_cambios", index=False)

print(f"Archivo generado: {salida}")

# Yo intento descargar el Excel automáticamente cuando el notebook se ejecuta en Google Colab.
try:
    from google.colab import files
    files.download(salida)
except Exception:
    pass

Archivo generado: tablas_PSP_4_4_2.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>